In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# imports
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance

In [ ]:
# options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
# let's read in the data and take a look
X_full = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv', index_col='Id')
X_test_full = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv', index_col='Id')
X_full.head()

In [ ]:
X_test_full.head()

In [ ]:
# check shapes for reference
print(X_full.shape)
print(X_test_full.shape)

In [ ]:
# let's split the training data into our data and target
X_full.dropna(subset='SalePrice', axis=0, inplace=True)
y = np.log(X_full['SalePrice']) # log for evaluation
X_full.drop('SalePrice', axis=1, inplace=True)

In [ ]:
# we could use train_test_split, but i am going to use CV

In [ ]:
# let's split up our categorical and numerical columns

ordinal_qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

ordinal_cols = ordinal_qual_cols

onehot_cols = [col for col in X_full.columns if 
                   (X_full[col].dtype == 'object') and 
                   (col not in ordinal_cols)]

categorical_cols = ordinal_cols + onehot_cols

numerical_cols = [col for col in X_full.columns if 
                  (X_full[col].dtype in ['int64','float64'])]

print(categorical_cols)
print(numerical_cols)

cols = categorical_cols + numerical_cols

X_train = X_full[cols].copy()
X_test = X_test_full[cols].copy()

In [ ]:
# let's figure out how we want to process our data
X_train.head()

In [ ]:
# we can likely let one hot encoding handle all of these nulls
# X_train[categorical_cols].isna().sum()

In [ ]:
# LotFrontage, GarageYrBlt, and MasVnrArea only ones with na. lets look further in
# X_train[numerical_cols].isna().sum()

In [ ]:
# X_train[X_train['LotFrontage'].isna()].head()
# X_train[X_train.LotFrontage == 0]
# it seems like any NA values in lotfrontage is if the house doesn't have any
# we can set this column's na values to 0 most likely

In [ ]:
# X_train[X_train['GarageYrBlt'].isna()].head()
# it seems like any na values in this column are when there is no garage
# X_train.head()
# the garage dates correlate pretty well with when the house was built
# i think dropping the garage year built and instead adding a hasGarage will be best

In [ ]:
# X_train[X_train['MasVnrArea'].isna()].head()
# it seems like an NaN value means there is not one, so we can input 0 for the missing values here

In [ ]:
# no cardinality is too high, so no need to drop any categorical
# [X_train[col].nunique() for col in X_train[categorical_cols].columns]

In [ ]:
# let's look into the testing data too to make sure these transformations are ok
# the one hot seems like it will work here, but a has pool feature should be added
# X_test[categorical_cols].isna().sum()

In [ ]:
# it appears that all of these NA values would also likely be best replaced with a 0
# X_test[numerical_cols].isna().sum()

In [ ]:
# the cardinalities are roughly the same, so one hot will still be the move
# [X_test[col].nunique() for col in X_test[categorical_cols].columns]

In [ ]:
# now that i know what the data is like, i will start feature engineering

for df in [X_train, X_test]:
    df['TotalSF'] = (df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF'])
    df['TotalBath'] = (df['BsmtHalfBath'] * .5 + df['BsmtFullBath'] + df['FullBath'] + df['HalfBath'] * .5)
    df['SellAge'] = (df['YrSold'] - df['YearBuilt'])
    df['OverallRating'] = (df['OverallQual'] + df['OverallCond'])

if 'TotalSF' not in numerical_cols:
    numerical_cols.extend(['TotalSF',
                           'TotalBath',
                           'SellAge',
                           'OverallRating',
                          ])

if 'GarageYrBlt' in numerical_cols:
    numerical_cols.remove('GarageYrBlt')
    numerical_cols.remove('BsmtFullBath')
    numerical_cols.remove('BsmtHalfBath')
    numerical_cols.remove('FullBath')
    numerical_cols.remove('HalfBath')

categorical_cols = ordinal_cols + onehot_cols
cols = categorical_cols + numerical_cols
X_train = X_train[cols].copy()
X_test = X_test[cols].copy()
X_train.head()

In [ ]:
# i can begin creating the pipeline now

quality_order = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']

ordinal_qual_transformer = Pipeline(steps=[
    ('ordqual', OrdinalEncoder(categories = [quality_order]*len(ordinal_qual_cols), handle_unknown='use_encoded_value', unknown_value=-1))
])

onehot_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numerical_transformer = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='constant', fill_value=0))
])

preprocessor = ColumnTransformer(transformers=[
    ('ordq', ordinal_qual_transformer, ordinal_qual_cols),
    ('oh', onehot_transformer, onehot_cols),
    ('num', numerical_transformer, numerical_cols)
])

In [ ]:
# it is time to create the model in the pipeline

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(#n_estimators=450, 
                           #max_depth=2, 
                           #learning_rate=.2, 
                           #colsample_bytree=.6,
                           #alpha=.1,
                           random_state=1))
])

In [ ]:
# now i will use grid search to find the best parameters for the model

param_grid={
    'model__n_estimators':[621],
    'model__max_depth':[2],
    'model__learning_rate':[.125],
    'model__colsample_bytree':[.525],
    'model__alpha':[.125],
    'model__lambda':[1.1],
}

#n_estimators=621, max_depth=2, learning_rate=.125, colsample_bytree=.525, alpha=.125, lambda=1.1

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y)
best_pipeline = grid_search.best_estimator_
grid_search.best_params_

In [ ]:
# use cv to see how strong pipeline is

cv_scores = -1 * cross_val_score(
    best_pipeline,
    X_train,
    y,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

print("RMSE per fold:", cv_scores)
print("Average RMSE:", cv_scores.mean())
# 0.11641603498024163

Best submitted - 0.1195547647350387: bathrooms in df, params n_estimators=450, max_depth=2, learning_rate=.2, colsample_bytree=.6, alpha=.1

Best found - 0.11641603498024163: bathrooms out of df, params n_estimators=621, max_depth=2, learning_rate=.125, colsample_bytree=.525, alpha=.125, lambda=1.1

In [ ]:
# using permutation importance to see what the most/least important features are

result = permutation_importance(best_pipeline,
                                X_train,
                                y,
                                n_repeats=5,
                                random_state=1,
                                scoring='neg_root_mean_squared_error'
                               )

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
}).sort_values(by='importance_mean', ascending=False)

print(importance_df.head(10))
print(importance_df.tail(10))

In [ ]:
# once satisfied with the MAE, we can predict the testing data and submit
best_pipeline.fit(X_train, y)

predictions = np.exp(best_pipeline.predict(X_test)) # re-exponent from beginning log

submission = pd.DataFrame({
    'Id':X_test.index,
    'SalePrice':predictions
})
submission.to_csv('submission.csv', index=False)
pd.read_csv('submission.csv').head()